In [9]:
import os
import re
import time
import torch
from PIL import Image
from diffusers import StableDiffusionPipeline
from transformers import CLIPTokenizer, CLIPTextModel

In [10]:
# Image output directory
output_dir = "generated_images"
os.makedirs(output_dir, exist_ok=True)

# ----------------------------
# Device selection: MPS (Apple Silicon), else CPU
# ----------------------------

if torch.backends.mps.is_available():
    device = "mps"  # macOS with Apple Silicon
elif torch.cuda.is_available():
    device = "cuda"
else:
    device = "cpu"  # CPU fallback for all platforms

print(f"Using device: {device}")

Using device: cuda


In [12]:
pipe = StableDiffusionPipeline.from_pretrained(
    "CompVis/stable-diffusion-v1-4",
    torch_dtype=torch.float32,  # float32 for MPS or CPU (no float16 on Mac)
    safety_checker=None,         # Disable NSFW filter; set to True if needed
)
tokenizer = CLIPTokenizer.from_pretrained("openai/clip-vit-large-patch14")
text_encoder = CLIPTextModel.from_pretrained("openai/clip-vit-large-patch14")

Loading pipeline components...: 100%|██████████| 6/6 [00:00<00:00, 11.48it/s]
You have disabled the safety checker for <class 'diffusers.pipelines.stable_diffusion.pipeline_stable_diffusion.StableDiffusionPipeline'> by passing `safety_checker=None`. Ensure that you abide to the conditions of the Stable Diffusion license and do not expose unfiltered results in services or applications open to the public. Both the diffusers team and Hugging Face strongly recommend to keep the safety filter enabled in all public facing circumstances, disabling it only for use-cases that involve analyzing network behavior or auditing its results. For more information, please have a look at https://github.com/huggingface/diffusers/pull/254 .


In [13]:
pipe.to(device)
text_encoder.to(device)

CLIPTextModel(
  (text_model): CLIPTextTransformer(
    (embeddings): CLIPTextEmbeddings(
      (token_embedding): Embedding(49408, 768)
      (position_embedding): Embedding(77, 768)
    )
    (encoder): CLIPEncoder(
      (layers): ModuleList(
        (0-11): 12 x CLIPEncoderLayer(
          (self_attn): CLIPSdpaAttention(
            (k_proj): Linear(in_features=768, out_features=768, bias=True)
            (v_proj): Linear(in_features=768, out_features=768, bias=True)
            (q_proj): Linear(in_features=768, out_features=768, bias=True)
            (out_proj): Linear(in_features=768, out_features=768, bias=True)
          )
          (layer_norm1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (mlp): CLIPMLP(
            (activation_fn): QuickGELUActivation()
            (fc1): Linear(in_features=768, out_features=3072, bias=True)
            (fc2): Linear(in_features=3072, out_features=768, bias=True)
          )
          (layer_norm2): LayerNorm((768,), ep

In [14]:
output_dir = "generated_images"
os.makedirs(output_dir, exist_ok=True)

In [36]:
# Set your text prompt here
prompt = "Elephant wearing glasses and the dog playing with ball next to it"

In [15]:
prompts = [
    "A cat sitting on a windowsill wearing glasses.",
    "An elephant balancing on a ball in a circus tent.",
    "A dog chasing a ball in a sunny park.",
    "A pair of glasses resting on a sleeping dog.",
    "A cat and a dog watching an elephant from a distance.",
    "A curious elephant sniffing a cat wearing glasses.",
    "A group of dogs playing soccer with a ball.",
    "A cat jumping over a dog lying next to a pair of glasses.",
    "An elephant wearing funny glasses in the jungle.",
    "A cat, a dog, and an elephant sharing a picnic.",
    "A dog wearing glasses and sitting on a skateboard.",
    "An elephant standing beside a giant rubber ball.",
    "A ball rolling between a cat and an elephant.",
    "A small dog trying to reach a ball on a high shelf.",
    "A fluffy cat curled up next to a pair of sunglasses.",
    "A basketball game between a cat and a dog.",
    "A cat sitting on top of a large elephant.",
    "An elephant playing with a dog and a ball.",
    "A dog wearing reading glasses and reading a book.",
    "A pair of glasses lying beside a toy elephant.",
    "A cat balancing on a red ball in a living room.",
    "An elephant gently nudging a ball with its trunk.",
    "A dog, a cat, and an elephant lined up for a race.",
    "Glasses perched on the nose of a very patient cat.",
    "A playful dog chasing a cat wearing tiny glasses.",
    "An elephant trying to wear glasses with its trunk.",
    "A cat watching an elephant bounce a ball.",
    "A dog, a ball, and a pair of glasses on a sandy beach.",
    "An elephant sitting in a chair next to a dog.",
    "A cat knocking glasses off a table onto a dog.",
    "A dog lying under a tree wearing cool sunglasses.",
    "A giant ball rolling towards a surprised cat.",
    "A kitten batting at a ball near an elephant's foot.",
    "A dog barking at a cat perched on an elephant.",
    "Glasses floating in a bowl next to a curious dog.",
    "A cartoon elephant spinning a basketball.",
    "A dog holding a ball in its mouth and wearing glasses.",
    "A cat reading a book through thick glasses.",
    "An elephant and a dog playing fetch with a ball.",
    "A pair of glasses lost in a jungle with an elephant.",
    "A dog jumping through a hoop held by a cat.",
    "An elephant standing on a ball, with a dog on its back.",
    "A sleepy cat resting on an elephant's back.",
    "A dog lying beside a ball and some broken glasses.",
    "An elephant walking next to a child holding a cat.",
    "A group of animals wearing glasses, including a dog.",
    "A cat wearing round glasses and sipping tea.",
    "An elephant balancing glasses on its trunk.",
    "A ball flying past a cat and hitting a dog.",
    "A stylish dog in glasses posing next to a cat.",
    "A dog trying to catch a ball thrown by an elephant.",
    "A cat hiding in a pile of glasses and books.",
    "A parade led by an elephant followed by dogs and cats.",
    "A kitten climbing onto a dog wearing glasses.",
    "A dog sitting beside a cat wearing sunglasses.",
    "A ball surrounded by cats, dogs, and elephants.",
    "A mischievous cat stealing a dog's glasses.",
    "A cat lounging on a ball, ignoring the elephant.",
    "A dog throwing a ball at a cat wearing glasses.",
    "A pair of glasses placed on an elephant statue.",
    "An elephant and a dog fighting over a tennis ball.",
    "A cat in glasses holding a ball of yarn.",
    "A dog splashing in water, chasing a floating ball.",
    "A sleepy elephant with glasses hanging from its trunk.",
    "A cat walking on a tightrope above a dog and a ball.",
    "A dog running away from a bouncing elephant.",
    "An elephant juggling glasses, balls, and cats.",
    "A dog playing fetch with an elephant.",
    "A cat wearing sunglasses on a sunny windowsill.",
    "A dog hiding behind a stack of books and glasses.",
    "An elephant, a dog, and a cat sitting in a classroom.",
    "A soccer game with a cat as the goalie.",
    "An elephant stepping carefully over a tiny ball.",
    "A dog rolling in grass next to a pair of glasses.",
    "A cat batting a tennis ball down the hallway.",
    "A dog in glasses typing on a tiny laptop.",
    "An elephant dancing on a stage wearing glasses.",
    "A kitten rolling a ball across the floor.",
    "A dog jumping on a trampoline next to a cat.",
    "An elephant watching a movie with 3D glasses.",
    "A cat resting inside a dog’s empty food bowl.",
    "A pair of sunglasses on a ball near a dog.",
    "A dog and a cat posing with glasses for a photo.",
    "An elephant holding a cat and a ball in its trunk.",
    "A dog peeking through a pair of broken glasses.",
    "A cat standing next to an elephant under a tree.",
    "A ball bouncing between a dog’s legs.",
    "A cartoon dog wearing giant novelty glasses.",
    "A cat snuggled up in a blanket with a dog.",
    "A dog looking into a mirror wearing glasses.",
    "An elephant surrounded by colorful balls.",
    "A dog chasing an elephant through a grassy field.",
    "A cat balancing a ball on its head.",
    "A dog wearing glasses and a party hat.",
    "A quiet elephant watching a cat sleep.",
    "A ball sitting between a cat’s paws.",
    "A dog wearing glasses and eating ice cream.",
    "An elephant reading a book to a cat and a dog.",
    "A cat curled up next to a ball and glasses.",
    "A dog wearing reading glasses and looking serious.",
    "A dog, a ball, a cat, an elephant – all in one picture.",
    "A ball under an elephant’s foot next to a cat.",
    "A pair of glasses stuck on a dog’s nose.",
    "A cat and a dog napping while holding a ball.",
    "An elephant balancing a cat and dog on its back."
]


In [16]:
def sanitize_filename(text):
    return re.sub(r"[^\w\-_\. ]", "_", text).strip().replace(" ", "_")

In [18]:
for prompt in prompts:
    base_filename = sanitize_filename(prompt)
    timestamp = time.strftime("%Y%m%d-%H%M%S")
    session_id = f"{base_filename}_{timestamp}"
    session_dir = os.path.join(output_dir, session_id)
    os.makedirs(session_dir, exist_ok=True)
    result = pipe(prompt)
    image = result.images[0]
    image_path = os.path.join(session_dir, "image.png")
    image.save(image_path)
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        clip_output = text_encoder(**inputs)
        text_embedding = clip_output.last_hidden_state  # (1, seq_len, hidden_dim)

    # Save embedding
    embedding_path = os.path.join(session_dir, "clip_embedding.pt")
    torch.save(text_embedding.cpu(), embedding_path)

100%|██████████| 50/50 [11:28<00:00, 13.77s/it]


In [ ]:
result = pipe(prompt)
image = result.images[0]

# Save image
image_path = os.path.join(session_dir, "image.png")
image.save(image_path)
print(f"Image saved to: {image_path}")

100%|██████████| 50/50 [12:19<00:00, 14.80s/it]


Image saved to: generated_images/Elephant_wearing_glasses_and_the_dog_playing_with_ball_next_to_it_20250604-111728/image.png


In [26]:
image.show()

In [39]:
inputs = tokenizer(prompt, return_tensors="pt").to(device)
with torch.no_grad():
    clip_output = text_encoder(**inputs)
    text_embedding = clip_output.last_hidden_state  # (1, seq_len, hidden_dim)

# Save embedding
embedding_path = os.path.join(session_dir, "clip_embedding.pt")
torch.save(text_embedding.cpu(), embedding_path)
print(f"CLIP embedding saved to: {embedding_path}")

CLIP embedding saved to: generated_images/Elephant_wearing_glasses_and_the_dog_playing_with_ball_next_to_it_20250604-111728/clip_embedding.pt
